# 02 · Framing y Protocolo

**Objetivo del notebook**: convertir un stream de bytes (TCP) en un stream de **mensajes tipados** que la aplicación pueda procesar.

Cubre las capas 3 y 4. Fuentes: [src/framing.ts](../../src/framing.ts), [src/protocol.ts](../../src/protocol.ts).

## 1. El problema: TCP no entiende de mensajes

TCP garantiza:

- entrega (con retransmisión)
- orden
- sin duplicados

Pero **no** preserva los límites entre escrituras:

```
Emisor:                  Receptor (cualquiera de estas):
  write("hola")            read("holamundo")
  write("mundo")           read("hol") + read("amundo")
                           read("hola") + read("mundo")
```

El kernel agrupa/fragmenta a su antojo. La aplicación necesita **decidir** dónde acaba un mensaje.

## 2. Soluciones históricas

### A. Separadores

Usar un byte/carácter que marca fin de mensaje. Ej: HTTP/1.1 con `\r\n\r\n`, SMTP con `\r\n.\r\n`.

Problemas:
- Hay que **escapar** el separador si aparece en el payload.
- Escanear byte a byte buscando el separador es O(N) por mensaje.
- No funciona si el payload es binario arbitrario.

### B. Prefijo de longitud (length-prefix)

Cada mensaje va precedido por **cuántos bytes mide** (típicamente un entero de 32 bits).

```
┌──────────────┬──────────────────────────────┐
│ length (u32) │ payload (length bytes)       │
└──────────────┴──────────────────────────────┘
```

Ventajas:
- Sin escapado: el payload es binario opaco.
- O(1) para saber cuánto queda por leer.
- Permite acotar tamaño máximo (defensa DoS).

Es lo que usa p2p-chat. También lo usan WebSocket, BitTorrent, gRPC...

## 3. Parser stateful

El receptor acumula chunks en un buffer y va sacando mensajes completos. Hay dos estados:

- **Buscando header**: `expected = None`. Si hay ≥4 bytes → leer length, pasar a siguiente estado.
- **Buscando payload**: `expected = N`. Si hay ≥N bytes → emitir mensaje, volver al estado anterior.

Reimplementémoslo en Python para verlo claro:

In [ ]:
import struct

class FrameParser:
    def __init__(self):
        self.buf = b''
        self.expected = None  # None = aún no sé cuánto leer; int = bytes restantes
    
    def push(self, chunk):
        self.buf += chunk
        msgs = []
        while True:
            if self.expected is None:
                if len(self.buf) < 4:
                    break  # falta header
                self.expected = struct.unpack('>I', self.buf[:4])[0]  # big-endian u32
                self.buf = self.buf[4:]
            if len(self.buf) < self.expected:
                break  # falta payload
            msgs.append(self.buf[:self.expected])
            self.buf = self.buf[self.expected:]
            self.expected = None
        return msgs

def frame(payload):
    return struct.pack('>I', len(payload)) + payload

# Test: framear 3 mensajes, partirlos en chunks irregulares, ver que el parser los reconstruye
wire = frame(b'hola') + frame(b'mundo') + frame(b'!')
print('en el cable:', wire)

parser = FrameParser()
# Fragmentación arbitraria: 3 bytes, luego 7, luego el resto
for chunk in [wire[:3], wire[3:10], wire[10:]]:
    print(f'push({chunk!r}) → mensajes: {parser.push(chunk)}')

## 4. Diseño defensivo: tamaño máximo

Sin límite, un atacante puede mandar un length de `0xFFFFFFFF` (4 GB) y forzarnos a reservar memoria hasta OOM. La defensa es trivial:

```ts
if (this.expected > MAX_FRAME) {
  throw new Error(`Frame too large: ${this.expected} > ${MAX_FRAME}`);
}
```

En p2p-chat `MAX_FRAME = 4 MB`. Una SDP de WebRTC + candidatos ICE pesa < 4 KB; un CHAT < 1 KB. 4 MB es enorme margen.

## 5. Capa 4: tipos de mensaje

Ya tenemos *frames* completos. Queda decidir qué hay dentro. En p2p-chat: **el primer byte del payload es el tipo**.

```
┌──────────────┬──────┬───────────────────────┐
│ length (u32) │ type │ payload JSON          │
│              │ u8   │ (omitido el campo type)│
└──────────────┴──────┴───────────────────────┘
```

Tabla completa:

In [ ]:
MSG = {
    0x01: ('HELLO',      'JSON {peerId, version}'),
    0x02: ('CHAT',       'JSON {messageId, text, ts}'),
    0x03: ('CHAT_ACK',   'JSON {messageId}'),
    0x04: ('BYE',        '(vacío)'),
    0x05: ('PING',       'JSON {nonce}'),
    0x06: ('PONG',       'JSON {nonce}'),
    0x10: ('CALL_OFFER', 'JSON {callId, sdp}'),
    0x11: ('CALL_ANSWER','JSON {callId, sdp}'),
    0x12: ('CALL_ICE',   'JSON {callId, candidate}'),
    0x13: ('CALL_END',   'JSON {callId, reason?}'),
}
for k, (name, payload) in MSG.items():
    print(f'  0x{k:02X}  {name:<13s}  {payload}')

### Por qué dejamos huecos en el hex

Los tipos están agrupados por familia con gaps deliberados:

- `0x01–0x04`: handshake + mensajería
- `0x05–0x06`: liveness
- `0x10–0x13`: señalización WebRTC

Si mañana añadimos `CHAT_EDIT` no nos veremos forzados a meterlo en `0x14` (donde rompe la lectura visual). Le damos `0x07` o `0x08` y queda en su familia. Reordenar números rompe todos los peers en producción.

Lección: **los IDs binarios son contratos**. Reserva rangos por familia y deja gaps.

## 6. Encode / decode

Codec simétrico. Encode:

```ts
function encode(msg) {
  const t = Buffer.from([msg.type]);
  if (msg.type === MSG.BYE) return t;     // sin payload
  const { type, ...rest } = msg;          // omitimos type del JSON
  return Buffer.concat([t, Buffer.from(JSON.stringify(rest))]);
}
```

Decode:

```ts
function decode(buf) {
  const type = buf[0];
  const body = buf.subarray(1);
  if (type === MSG.BYE) return { type };
  return { type, ...JSON.parse(body.toString('utf8')) };
}
```

Detalles importantes:

1. **Omitir `type` del JSON**: ya va en el primer byte. Si lo dejas, duplicas info en cada mensaje y pagas bytes innecesariamente.
2. **Tipo desconocido → lanzar excepción**: en transport.ts esto se captura y cierra el socket. "Mensaje basura" trata como peer hostil o incompatible.
3. **JSON parse falla → idem**: misma defensa.

## 7. ¿Y si tuviéramos payload binario?

p2p-chat no transfiere ficheros — todo lo que viaja por TCP es texto (JSON). Pero hay un caso futuro: el **ejercicio de file transfer**. Ahí necesitarás mandar bloques binarios.

Opciones:

1. **Base64 dentro del JSON**: simple, +33% de overhead.
2. **Híbrido**: 1 byte tipo + N bytes header JSON + 4 bytes longitud blob + blob crudo.
3. **Binario puro**: protocol buffers, MessagePack, BSON.

Para 64 KB por chunk, la opción 1 es la más sencilla; lo verás en [EXERCISES.md](../EXERCISES.md).

## 8. Siguiente paso

Tenemos mensajes tipados viajando entre peers. En el notebook 03 vemos qué se puede construir encima: **ACKs con timeout, historial persistente y RTT por heartbeat**.